## 골든셋으로 정확도 측정

In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day01" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w7" / "day01"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

SAMPLES = ROOT / "sandbox" / "w5" / "day01" / "samples"
FORMATS = SAMPLES / "_formats"

프로젝트 루트  : /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice


In [3]:
if str(ROOT / "evals") not in sys.path:
    sys.path.insert(0, str(ROOT / "evals"))
import run_eval

THRESHOLD = 0.55       

def hit(score: float) -> list[dict]:
    return [{"doc_id": "DOC-HR-014", "version": "v2.0", "kind": "조항",
             "locator": "제9조(일비)", "text": "일비는 1일 30,000원으로 한다",
             "score": score}]

normal = {"id": "G01", "expect_doc": "DOC-HR-014", "expect_version": "v2.0",
          "expect_keywords": ["30,000", "일비"]}
unanswerable = {"id": "G25", "unanswerable": True}

for label, item, hits in [("보통 문항 · 점수 낮음", normal, hit(0.31)),
                          ("unanswerable · 점수 낮음", unanswerable, hit(0.31)),
                          ("unanswerable · 점수 높음", unanswerable, hit(0.82)),
                          ("보통 문항 · 정답 문서 상위", normal, hit(0.82))]:
    ok, why = run_eval.judge(item, hits, THRESHOLD)
    print(f"{label:26} → {str(ok):6} {why}")

보통 문항 · 점수 낮음              → False  임계값 미달(0.31
unanswerable · 점수 낮음       → True   근거 부족 정상
unanswerable · 점수 높음       → False  임계값 초과(0.82) - 지어낼 위험
보통 문항 · 정답 문서 상위           → True   DOC-HR-014 v2.0 - 제9조(일비)


In [4]:
import importlib
import statistics
from datetime import datetime
import yaml

import run_eval

run_eval = importlib.reload(run_eval)

items = yaml.safe_load(run_eval.GOLDEN.read_text(encoding="utf-8"))["items"]


if not list(run_eval.REPORTS.glob("eval_*.json")):
    print("리포트가 없어 여기서 한 번 대신 재보기 — 수업에서는 위 명령이 정본")
    run_eval.save({"measured_at": datetime.now().isoformat(timespec="seconds"),
                   "threshold": run_eval.settings.retrieval_threshold,
                   "top_n": run_eval.DEFAULT_TOP_N,
                   "runs": [run_eval.run_config(run_eval.CONFIGS[0], items,
                                                run_eval.DEFAULT_TOP_N)]})

with run_eval.session_scope() as session:
    bests = [max((h["score"] for h in run_eval.retrieve(session, i["question"],
                                                        run_eval.DEFAULT_TOP_N)),
                 default=0.0)
             for i in items]

th = run_eval.settings.retrieval_threshold
print(f"best 점수 — 최소 {min(bests):.3f} · 중앙 {statistics.median(bests):.3f} "
      f"· 최대 {max(bests):.3f}")
print(f"임계값 {th} 를 넘은 문항 = {sum(1 for b in bests if b >= th)} / {len(bests)}")
print()
for cand in (0.10, 0.20, 0.30, 0.40, 0.55):
    print(f" threshold {cand:5.2f} : 넘는 문항 {sum(1 for b in bests if b >= cand)}")

11:41:22 INFO     sentence_transformers.base.model : No device provided, using mps
11:41:22 INFO     sentence_transformers.base.model : Loading SentenceTransformer model from ../models/bge-m3.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

11:41:24 INFO     app.integrations.local_embed : 로컬 임베딩 모델 준비: ../models/bge-m3 / 1024차원


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

best 점수 — 최소 0.430 · 중앙 0.578 · 최대 0.760
임계값 0.55 를 넘은 문항 = 21 / 30

 threshold  0.10 : 넘는 문항 30
 threshold  0.20 : 넘는 문항 30
 threshold  0.30 : 넘는 문항 30
 threshold  0.40 : 넘는 문항 30
 threshold  0.55 : 넘는 문항 21


In [5]:
import importlib
from starlette.testclient import TestClient
import app.main

importlib.reload(app.main)
fastapi_app = app.main.app
ops_paths = [p for p in fastapi_app.openapi()['paths'] if '/ops' in p]
print('등록된 ops 경로 :', ops_paths)

res = TestClient(fastapi_app).get('/api/v1/ops/eval-runs')
rows = res.json() if res.status_code == 200 else []
print(f'GET /api/v1/ops/eval-runs → {res.status_code} · 행 {len(rows)}개')

for row in rows:
    print(f"  {row['config']}  ord={row['ord']}  accuracy={row['accuracy']}  item_count={row['item_count']}  measured_at={row['measured_at']}")


등록된 ops 경로 : ['/api/v1/ops/eval-runs']
12:11:41 INFO     httpx2 : HTTP Request: GET http://testserver/api/v1/ops/eval-runs "HTTP/1.1 200 OK"
GET /api/v1/ops/eval-runs → 200 · 행 1개
  기본 검색  ord=0  accuracy=80.0  item_count=30  measured_at=2026-10-12T12:03:49.129751+09:00


In [6]:
import json
import re

REPORTS = ROOT / 'evals' / 'reports'
latest = sorted(REPORTS.glob('eval_*.json'))[-1]
results = json.loads(latest.read_text(encoding='utf-8'))['runs'][0]['results']
wrong = [r for r in results if not r['ok']]
print(f'틀린 문항 {len(wrong)}개\n')

for r in wrong:
    print(f"{r['id']:<5}{r['type']:<14}{r['why']:<26}{r['question'][:12]}…")
print('\n사유별')

counts: dict[str, int] = {}
for r in wrong:
    label = re.sub('\\(.*?\\)', '', r['why']).strip()
    counts[label] = counts.get(label, 0) + 1
for (label, n) in sorted(counts.items(), key=lambda kv: -kv[1]):
    print(f'  {label:<22}{n:>3}')


틀린 문항 6개

G07  단일 조항         정답 문서가 상위에 없음             법인카드는 어떤 경우에…
G16  표 안의 조건 조합    정답 문서가 상위에 없음             직급별 일비를 표로 정…
G23  타 부서 문서 참조    정답 문서가 상위에 없음             협력사 평가는 어떤 기…
G24  타 부서 문서 참조    정답 문서가 상위에 없음             접근권한은 누가 승인하…
G25  규정에 없는 사항     임계값 초과(0.56) - 지어낼 위험     해외 출장 시 비즈니스…
G26  규정에 없는 사항     임계값 초과(0.60) - 지어낼 위험     해외 체류비는 하루에 …

사유별
  정답 문서가 상위에 없음           4
  임계값 초과 - 지어낼 위험         2


In [7]:
from sqlalchemy import select
from app.db.session import session_scope
from app.models.eval import EvalWeakType
by_type = json.loads(latest.read_text(encoding='utf-8'))['runs'][0]['by_type']

def top_reason(label: str) -> str:
    reasons = [re.sub('\\(.*?\\)', '', r['why']).strip() for r in wrong if r['type'] == label]
    return max(set(reasons), key=reasons.count) if reasons else '오답 없음'

DIAGNOSIS = {'단일 조항': top_reason('단일 조항'), '표 안의 조건 조합': '표가 조항 단위로 잘림', '타 부서 문서 참조': '보안등급 정상 차단 · 시험지 오류 2건', '규정에 없는 사항': '근거부족 정상', '개정 전후 비교': '과거 버전 조회 경로'}
ranked = sorted(by_type.items(), key=lambda kv: (kv[1]['ok'] / kv[1]['n'], list(DIAGNOSIS).index(kv[0])))

with session_scope() as s:
    for old in s.execute(select(EvalWeakType)).scalars().all():
        s.delete(old)
    for (i, (label, stat)) in enumerate(ranked[:4]):
        pct = '—' if label == '규정에 없는 사항' else f"{stat['ok'] / stat['n'] * 100:.1f}"
        s.add(EvalWeakType(ord=i, label=label, count=stat['n'], accuracy=pct, diagnosis=DIAGNOSIS[label]))

with session_scope() as s:
    saved = s.execute(select(EvalWeakType.ord, EvalWeakType.label, EvalWeakType.count, EvalWeakType.accuracy, EvalWeakType.diagnosis).order_by(EvalWeakType.ord)).all()
print(f'eval_weak_types {len(saved)}줄\n')
print(' ord | label                | count | accuracy | diagnosis')
print('-----+----------------------+-------+----------+------------------------------')
for (ord_, label, cnt, acc, diag) in saved:
    print(f'{ord_:>4} | {label:<20} | {cnt:>5} | {acc:<8} | {diag}')


eval_weak_types 4줄

 ord | label                | count | accuracy | diagnosis
-----+----------------------+-------+----------+------------------------------
   0 | 타 부서 문서 참조           |     4 | 50.0     | 보안등급 정상 차단 · 시험지 오류 2건
   1 | 규정에 없는 사항            |     6 | —        | 근거부족 정상
   2 | 표 안의 조건 조합           |     6 | 83.3     | 표가 조항 단위로 잘림
   3 | 단일 조항                |    10 | 90.0     | 정답 문서가 상위에 없음
